In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import re

SOURCES = [
    "calcofi",
    "cce_moorings",
    "easyoneargo",
    "ca_beach_water_quality",
]

def load_raw_inventory(source_name: str):
    return (
        spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .load(f"s3://toxictide-raw/sources/{source_name}/raw/")
        .select(
            F.lit(source_name).alias("source_name"),
            "path",
            "length",
            "modificationTime"
        )
    )

def load_summary_inventory(source_name: str):
    df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .option("multiLine", "true")
        .json(f"s3://toxictide-raw/sources/{source_name}/summaries/")
    )
    # Keep the file payload's source_name if present, otherwise add one
    if "source_name" not in df.columns:
        df = df.withColumn("source_name", F.lit(source_name))
    return df

def load_manifest_inventory(source_name: str):
    df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .option("multiLine", "true")
        .json(f"s3://toxictide-raw/sources/{source_name}/manifests/")
    )
    if "source_name" not in df.columns:
        df = df.withColumn("source_name", F.lit(source_name))
    return df

def sanitize_column_name(col_name: str) -> str:
    c = col_name.strip().lower()
    c = re.sub(r"[^a-z0-9_]", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    if not c:
        c = "col"
    if c[0].isdigit():
        c = f"col_{c}"
    return c

def sanitize_columns(df):
    seen = set()
    exprs = []
    mapping = {}

    for c in df.columns:
        safe = sanitize_column_name(c)
        base = safe
        i = 2
        while safe in seen:
            safe = f"{base}_{i}"
            i += 1
        seen.add(safe)
        mapping[c] = safe
        exprs.append(F.col(f"`{c}`").alias(safe))

    return df.select(*exprs), mapping

In [0]:
raw_inventory = None

for src in SOURCES:
    df = load_raw_inventory(src)
    raw_inventory = df if raw_inventory is None else raw_inventory.unionByName(df)

display(raw_inventory.orderBy(F.desc("length")))

source_name,path,length,modificationTime
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/127233.tar.gz,7969262537,2026-04-19T08:00:49.000Z
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/126471.tar.gz,7907834785,2026-04-19T08:00:34.000Z
ca_beach_water_quality,s3://toxictide-raw/sources/ca_beach_water_quality/raw/run_date=2026-04-19/monitoring_results/beach_monitoring_results.csv,1755348817,2026-04-19T08:23:27.000Z
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/127234.tar.gz,1595051070,2026-04-19T07:50:26.000Z
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/126470.tar.gz,1586383338,2026-04-19T07:50:47.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_11_D_Meteorology-Wind.nc,84858112,2026-04-19T03:42:44.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce2/OS_CCE2_11_D_Meteorology-Wind.nc,77846924,2026-04-19T08:16:53.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce1/OS_CCE1_13_D_Meteorology-Wind.nc,64348520,2026-04-19T08:09:37.000Z
cce_moorings,s3://toxictide-raw/sources/cce_moorings/raw/run_date=2026-04-19/cce2/OS_CCE2_10_D_Meteorology-Wind.nc,51802496,2026-04-19T08:17:16.000Z
calcofi,s3://toxictide-raw/sources/calcofi/raw/run_date=2026-04-19/bottle_database/CalCOFI_Database_194903-202105_csv_16October2023.zip,37779228,2026-04-19T08:02:34.000Z


In [0]:
raw_inventory_summary = (
    raw_inventory
    .groupBy("source_name")
    .agg(
        F.count("*").alias("n_files"),
        F.sum("length").alias("total_bytes"),
        F.min("length").alias("min_bytes"),
        F.max("length").alias("max_bytes"),
    )
    .orderBy("source_name")
)

display(raw_inventory_summary)

source_name,n_files,total_bytes,min_bytes,max_bytes
ca_beach_water_quality,3,1786512559,649554,1755348817
calcofi,2,37782792,3564,37779228
cce_moorings,266,997160828,8192,84858112
easyoneargo,4,19058531730,1586383338,7969262537


In [0]:
(
    raw_inventory
    .write
    .mode("overwrite")
    .saveAsTable("toxictide.bronze.raw_file_inventory")
)

(
    raw_inventory_summary
    .write
    .mode("overwrite")
    .saveAsTable("toxictide.bronze.raw_file_inventory_summary")
)

In [0]:
summary_inventory = None

for src in SOURCES:
    try:
        df = load_summary_inventory(src)
        summary_inventory = df if summary_inventory is None else summary_inventory.unionByName(df, allowMissingColumns=True)
    except Exception as e:
        print(f"Skipping summary load for {src}: {e}")

display(summary_inventory)

Skipping summary load for cce_moorings: [COLUMN_ALREADY_EXISTS] The column `conventions` already exists. Choose another name or rename the existing column. SQLSTATE: 42711

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.columnAlreadyExistsError(QueryCompilationErrors.scala:4229)
	at org.apache.spark.sql.util.SchemaUtils$.checkColumnNameDuplication(SchemaUtils.scala:182)
	at org.apache.spark.sql.util.SchemaUtils$.checkSchemaColumnNameDuplication(SchemaUtils.scala:97)
	at org.apache.spark.sql.util.SchemaUtils$.$anonfun$checkSchemaColumnNameDuplication$2(SchemaUtils.scala:99)
	at org.apache.spark.sql.util.SchemaUtils$.$anonfun$checkSchemaColumnNameDuplication$2$adapted(SchemaUtils.scala:98)
	at scala.collection.ArrayOps$.foreach$extension(ArrayOps.scala:1324)
	at org.apache.spark.sql.util.SchemaUtils$.checkSchemaColumnNameDuplication(SchemaUtils.scala:98)
	at org.apache.spark.sql.util.SchemaUtils$.checkSchemaColumnNameDuplica

columns delimiter_used dtypes_preview encoding_used file_type filename kind preview sample_rows size_bytes source_group source_name source_url table_overview_error zip_overview netcdf_overview_error preview_lines remote_expected_size_bytes sample_lines table_overview List(Field Name, Units, Description) , List(object, object, object, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null) cp1252 csv Cast_Field_Descriptions.csv file List(List(Cast Count - All CalCOFI casts ever conducted, consecutively numbered, Cst_Cnt, n.a., null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null), List(Cruise identifier [Year]-[Month]-[Day]-C-[Ship Code], Cruise_ID, n.a., null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null), List(Cruise Name [Year][Month], Cruise, n.a., null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null), List(Cruise Name and Station [Year][Month][Line][Station], Cruz_Sta, n.a., null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 

In [0]:
print("summary_inventory columns:")
print(summary_inventory.columns)

display(
    summary_inventory.select(
        "source_name",
        "source_group",
        "filename",
        "file_type",
        "size_bytes"
    ).limit(20)
)

summary_inventory columns:
['columns', 'delimiter_used', 'dtypes_preview', 'encoding_used', 'file_type', 'filename', 'kind', 'preview', 'sample_rows', 'size_bytes', 'source_group', 'source_name', 'source_url', 'table_overview_error', 'zip_overview', 'netcdf_overview_error', 'preview_lines', 'remote_expected_size_bytes', 'sample_lines', 'table_overview']


source_name,source_group,filename,file_type,size_bytes
calcofi,ctd_cast_files,Cast_Field_Descriptions.csv,csv,3564
calcofi,ctd_cast_files,Cast_Field_Descriptions.csv,csv,3564
calcofi,ctd_cast_files,Cast_Field_Descriptions.csv,csv,3564
calcofi,bottle_database,CalCOFI_Database_194903-202105_csv_16October2023.zip,zip,37779228
calcofi,bottle_database,CalCOFI_Database_194903-202105_csv_16October2023.zip,zip,37779228
calcofi,bottle_database,CalCOFI_Database_194903-202105_csv_16October2023.zip,zip,37779228
calcofi,bottle_database,CalCOFI_Database_194903-202105_csv_16October2023.zip,zip,37779228
easyoneargo,null,126471.tar.gz,text,7907834785
easyoneargo,null,127233.tar.gz,text,7969262537
easyoneargo,null,127234.tar.gz,text,1595051070


In [0]:
(
    summary_inventory
    .write
    .mode("overwrite")
    .saveAsTable("toxictide.bronze.raw_summary_inventory")
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6904815970628396>, line 5
      1 (
      2     summary_inventory
      3     .write
      4     .mode("overwrite")
----> 5     .saveAsTable("toxictide.bronze.raw_summary_inventory")
      6 )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.u

In [0]:
calcofi_cast_fields = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("s3://toxictide-raw/sources/calcofi/raw/*/ctd_cast_files/*.csv")
)

display(calcofi_cast_fields.limit(50))
print("Columns:", calcofi_cast_fields.columns)

Field Name,Units,Description
Cst_Cnt,n.a.,"Cast Count - All CalCOFI casts ever conducted, consecutively numbered"
Cruise_ID,n.a.,Cruise identifier [Year]-[Month]-[Day]-C-[Ship Code]
Cruise,n.a.,Cruise Name [Year][Month]
Cruz_Sta,n.a.,Cruise Name and Station [Year][Month][Line][Station]
DbSta_ID,n.a.,Line and Station
Cast_ID,n.a.,Cast Identifier [Century] - [YY][MM][ShipCode] - [CastType][Julian Day] - [CastTime]-[Line][Sta]
Sta_ID,n.a.,Line and Station
Quarter,n.a.,Quarter of the year
Sta_Code,n.a.,Station Designation (See Station_ID and 0-St_Code for codes)
Distance,nautical miles,"Nautical miles from coast intercept, calculated from estimated latitude and longitude"


Columns: ['Field Name', 'Units', 'Description']


In [0]:
calcofi_summaries = (
    summary_inventory
    .filter(F.col("source_name") == "calcofi")
)

display(calcofi_summaries.select("source_group", "filename", "file_type", "size_bytes"))

source_name,filename,file_type,size_bytes
calcofi,Cast_Field_Descriptions.csv,csv,3564
calcofi,Cast_Field_Descriptions.csv,csv,3564
calcofi,Cast_Field_Descriptions.csv,csv,3564
calcofi,CalCOFI_Database_194903-202105_csv_16October2023.zip,zip,37779228
calcofi,CalCOFI_Database_194903-202105_csv_16October2023.zip,zip,37779228
calcofi,CalCOFI_Database_194903-202105_csv_16October2023.zip,zip,37779228
calcofi,CalCOFI_Database_194903-202105_csv_16October2023.zip,zip,37779228


In [0]:
import io
import zipfile

zip_path = "s3://toxictide-raw/sources/calcofi/raw/run_date=2026-04-19/bottle_database/CalCOFI_Database_194903-202105_csv_16October2023.zip"

zip_row = (
    spark.read.format("binaryFile")
    .load(zip_path)
    .select("path", "length", "content")
    .first()
)

zip_bytes = bytes(zip_row["content"])
zf = zipfile.ZipFile(io.BytesIO(zip_bytes))

zip_members = [(zi.filename, zi.file_size, zi.compress_size) for zi in zf.infolist()]
zip_members_df = spark.createDataFrame(zip_members, ["member_name", "file_size", "compressed_size"])

display(zip_members_df.orderBy(F.desc("file_size")))

member_name,file_size,compressed_size
CalCOFI_Database_194903-202105_csv_16October2023/CalCOFI_Database_194903-202105_csv_16October2023/194903-202105_Bottle.csv,175418694,35325993
CalCOFI_Database_194903-202105_csv_16October2023/CalCOFI_Database_194903-202105_csv_16October2023/194903-202105_Cast.csv,12525596,2452305
CalCOFI_Database_194903-202105_csv_16October2023/CalCOFI_Database_194903-202105_csv_16October2023/,0,0


In [0]:
ca_beach_stations = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("s3://toxictide-raw/sources/ca_beach_water_quality/raw/*/monitoring_stations/*.csv")
)

display(ca_beach_stations.limit(20))
print("Columns:", ca_beach_stations.columns)

Regional Board,Station_id,Station_Agency_id,BeachName_id,Station_Name,Station_Description,AgencyStationIdentifier,Station_UpperLat,Station_UpperLon,Station_LowerLat,Station_LowerLon,SwimSeason,OffSeason,AsNeededSeason,CountyCode,CountyName,Status,UpdateDate,Stations Insert Date,pointZero,Datum,Beach_AgencyName,Beach_Name,Description,BeachType,Beach Length,AttendanceWinter,AttendanceSummer,NearestCityName,WaterBodyName,WaterBodyClass,AB411Beach,USEPAID,BeachAccess,SwimSeasonLength,WaterBodyType,WaterShedName,Beach_UpperLat,Beach_UpperLon,Beach_LowerLat,Beach_LowerLon,Beach_County,Beach_User_id,Beach_Status,CountAsBeach,Agency_id,Agency_Code,Agency_Name,Agency_Jurisdiction,StreetAddress,City,Station Contact Person,Zip,EstMonCost,EstCoastlineAdvisoryCost,EstTotalCost,Station Comments,County,Agency_User_id,Regional Board Number,Regional Board Name
RB8,1,21,53,0,50\' N of Santa Ana River,0,33.62929,-117.95968,33.62929,-117.95968,Twice a week,Twice a week,Once per day,OC,Orange,Active,2026-02-09,InsertDate,1,4,Orange County Environmental Health Division,Huntington State Beach,Huntington State Beach,UNKNOWN,2.01,0,0,Huntington Beach,Pacific Ocean,Saltwater,No,CA857004,PUBLIC,12,Open Coast,Seal Beach,33.646021,-117.988024,33.629289,-117.959471,Orange,3,Active,1,21,EH,Orange County Environmental Health Division,County,2009 E. Edinger Avenue,Santa Ana,Michael Fennessy,92705,NULL,NULL,More than $250000,Michael Fennessy,Orange,1,8,Santa Ana
RB3,2,32,145,1000,Rincon Beach,1000,34.373398,-119.476088,34.373287201,-119.35891,Once a week,Once a week,Twice a week,VC,Ventura,Active,2015-08-25,InsertDate,1,4,Ventura County Environmental Health Division,Rincon Beach,Rincon Beach,UNKNOWN,0.35,0,0,Carpinteria,Pacific Ocean,Saltwater,No,CA527007,PUBLIC,12,Open Coast,Ventura River,34.3733,-119.4768,34.3759,-119.4715,Ventura,115,Active,1,32,VC,Ventura County Environmental Health Division,County,800 S. Victoria Ave.,Ventura,Richard Hauge,93009,NULL,NULL,$100000-$250000,DUNS=071818830,Ventura,1,3,Central Coast
RB4,3,32,148,10000,Solimar Beach,10000,34.310402,-119.35891,34.310402,-119.476524353,Once a week,Once a week,Twice a week,VC,Ventura,Active,2015-09-03,InsertDate,1,4,Ventura County Environmental Health Division,Solimar Beach,Solimar Beach,UNKNOWN,1.57,0,0,Solimar Beach,Pacific Ocean,Saltwater,No,CA844317,PUBLIC,12,Open Coast,Ventura River,34.32101,-119.37541,34.307619,-119.353167,Ventura,7,Active,1,32,VC,Ventura County Environmental Health Division,County,800 S. Victoria Ave.,Ventura,Richard Hauge,93009,NULL,NULL,$100000-$250000,DUNS=071818830,Ventura,1,4,Los Angeles
RB3,4,32,145,1001,Rincon Creek (mouth),1001,34.37351,-119.4756317,34.37351,-119.4756317,None(not sampled),None(not sampled),None(not sampled),VC,Ventura,Inactive,2015-09-03,InsertDate,1,4,Ventura County Environmental Health Division,Rincon Beach,Rincon Beach,UNKNOWN,0.35,0,0,Carpinteria,Pacific Ocean,Saltwater,No,CA527007,PUBLIC,12,Open Coast,Ventura River,34.3733,-119.4768,34.3759,-119.4715,Ventura,115,Active,1,32,VC,Ventura County Environmental Health Division,County,800 S. Victoria Ave.,Ventura,Richard Hauge,93009,NULL,NULL,$100000-$250000,DUNS=071818830,Ventura,1,3,Central Coast
RB3,5,32,145,1050,Rincon Beach,1050,34.3739,-119.4749451,34.3739,-119.4749451,None(not sampled),None(not sampled),None(not sampled),VC,Ventura,Inactive,2015-09-03,InsertDate,1,4,Ventura County Environmental Health Division,Rincon Beach,Rincon Beach,UNKNOWN,0.35,0,0,Carpinteria,Pacific Ocean,Saltwater,No,CA527007,PUBLIC,12,Open Coast,Ventura River,34.3733,-119.4768,34.3759,-119.4715,Ventura,115,Active,1,32,VC,Ventura County Environmental Health Division,County,800 S. Victoria Ave.,Ventura,Richard Hauge,93009,NULL,NULL,$100000-$250000,DUNS=071818830,Ventura,1,3,Central Coast
RB3,6,32,145,1100,Rincon Beach,1100,34.37538,-119.4730453,34.37538,-119.4730453,Once a week,None(not sampled),None(not sampled),VC,Ventura,Active,2015-09-09,InsertDate,1,4,Ventura County Environmental Health Division,Rincon Beach,Rincon B

Columns: ['Regional Board', 'Station_id', 'Station_Agency_id', 'BeachName_id', 'Station_Name', 'Station_Description', 'AgencyStationIdentifier', 'Station_UpperLat', 'Station_UpperLon', 'Station_LowerLat', 'Station_LowerLon', 'SwimSeason', 'OffSeason', 'AsNeededSeason', 'CountyCode', 'CountyName', 'Status', 'UpdateDate', 'Stations Insert Date', 'pointZero', 'Datum', 'Beach_AgencyName', 'Beach_Name', 'Description', 'BeachType', 'Beach Length', 'AttendanceWinter', 'AttendanceSummer', 'NearestCityName', 'WaterBodyName', 'WaterBodyClass', 'AB411Beach', 'USEPAID', 'BeachAccess', 'SwimSeasonLength', 'WaterBodyType', 'WaterShedName', 'Beach_UpperLat', 'Beach_UpperLon', 'Beach_LowerLat', 'Beach_LowerLon', 'Beach_County', 'Beach_User_id', 'Beach_Status', 'CountAsBeach', 'Agency_id', 'Agency_Code', 'Agency_Name', 'Agency_Jurisdiction', 'StreetAddress', 'City', 'Station Contact Person', 'Zip', 'EstMonCost', 'EstCoastlineAdvisoryCost', 'EstTotalCost', 'Station Comments', 'County', 'Agency_User_id',

In [0]:
ca_beach_postings = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("s3://toxictide-raw/sources/ca_beach_water_quality/raw/*/postings_closures/*.csv")
)

display(ca_beach_postings.limit(20))
print("Columns:", ca_beach_postings.columns)

DURATION CALCULATED,Regional Board,Advisory id,HistoricalStationName,prefix,mile,DateofAdvisory,TimeofAdvisory,DateOpened,TimeOpened,AdvisoryType,AdvisoryCause,Description,Extent,StartofExtent,StartOfExtentType,EndofExtent,EndOfExtentType,Source,Advisory Comments,DescriptionofArea,ActionBy,AffectedLengthUnit,SubstanceName,SubstanceVolume,SubstanceUnit,AdditionalAction,ActionDate,ActionDescription,County,User_id,Enterococcus,Fecal Coliforms,Total Coliforms,E.Coli,Ratio,Advisory Record Create Date,Station_id,Station_Agency_id,BeachName_id,Station_Name,Station_Description,AgencyStationIdentifier,Station_UpperLat,Station_UpperLon,Station_LowerLat,Station_LowerLon,SwimSeason,OffSeason,AsNeededSeason,CountyCode,CountyName,Status,UpdateDate,Advisories Insert Date,pointZero,Datum,BeachProfileId,Beach_AgencyName,Beach_Name,BeachDescription,BeachType,Beach Length,AttendanceWinter,AttendanceSummer,NearestCityName,WaterBodyName,WaterBodyClass,AB411Beach,USEPAID,BeachAccess,SwimSeasonLength,WaterBodyType,WaterShedName,Beach_UpperLat,Beach_UpperLon,Beach_LowerLat,Beach_LowerLon,Beach_County,Beach_User_id,Beach_Status,CountAsBeach,Agency_id,Agency_Code,Agency_Name,Agency_Jurisdiction,StreetAddress,City,Advisories Contact Person,Zip,EstMonCost,EstCoastlineAdvisoryCost,EstTotalCost,Agency_Comments,Agency_County,Agency_User_id,Regional Board Number,Regional Board Name
1,RB4,433,B-24,I,0,5/3/2010,14:35:33,5/4/2010,14:37:02,Posting,Unknown Cause,Colorado Lagoon-South,0.05681818,-0.02840909,Lat/Long,0.02840909,Lat/Long,Unknown,null,Long Beach Colorado Lagoon-South,null,miles,Unknown,null,Gallons,null,null,null,Long Beach City,5,1,1,1,0,1,9/19/2015 3:39:59 PM,110,4,455,B-24,Colorado Lagoon-South,B-24,33.77062,-118.13475,33.77062,-118.13475,Once a week,Once a week,Twice a week,LB,Long Beach City,Active,2016-02-18,2015-08-19,1,4,455,City of Long Beach Health and Human Services,Colorado Lagoon,Colorado Lagoon,ROCKY,0.1,0,0,LB,Pacific Ocean,Estuarine,No,CA699607,PUBLIC,12,"Sound, Bay, or Inlet",San Gabriel River,33.771323,-118.133145,33.770194,-118.132057,Long Beach City,5,Active,1,4,LB,City of Long Beach Health and Human Services,City,2525 Grand Ave,Long Beach,Hans Tritten,90815,NULL,NULL,NULL,Jackie Hampton,Los Angeles,1,4,Los Angeles
31,RB4,434,B-25,I,0,4/3/2010,14:35:51,5/4/2010,14:37:12,Posting,Unknown Cause,Colorado Lagoon-North,0.05681818,0.02249091,Lat/Long,0.07930909,Lat/Long,Unknown,null,Long Beach Colorado Lagoon-North,null,miles,Unknown,null,Gallons,null,null,null,Long Beach City,5,1,null,null,null,null,9/19/2015 3:39:59 PM,111,4,455,B-25,Colorado Lagoon-North,B-25,33.7712,-118.13435,33.7712,-118.13435,Once a week,Once a week,Twice a week,LB,Long Beach City,Active,2016-02-18,2015-08-19,1,4,455,City of Long Beach Health and Human Services,Colorado Lagoon,Colorado Lagoon,ROCKY,0.1,0,0,LB,Pacific Ocean,Estuarine,No,CA699607,PUBLIC,12,"Sound, Bay, or Inlet",San Gabriel River,33.771323,-118.133145,33.770194,-118.132057,Long Beach City,5,Active,1,4,LB,City of Long Beach Health and Human Services,City,2525 Grand Ave,Long Beach,Hans Tritten,90815,NULL,NULL,NULL,Jackie Hampton,Los Angeles,1,4,Los Angeles
1,RB4,435,B-5,C,3,5/17/2010,14:47:43,5/18/2010,15:06:03,Posting,Unknown Cause,5th Place-Beach,0.05681818,3.799591,Lat/Long,3.856409,Lat/Long,Unknown,null,Long Beach 5th Place-Beach,null,miles,Unknown,null,Gallons,null,null,null,Long Beach City,5,1,null,null,null,null,9/19/2015 3:39:59 PM,125,4,381,B-5,5th Place-Beach,B-5,33.76393,-118.1784286,33.76393,-118.1784286,Once a week,Once a week,Twice a week,LB,Long Beach City,Active,2016-02-18,2015-08-19,1,4,381,City of Long Beach Health and Human Services,Long Beach,Long Beach,UNKNOWN,3.7,0,0,LB,Pacific Ocean,Saltwater,No,CA279698,PUBLIC,12,Open Coast,San Gabriel River,33.74515,-118.119183,33.763467,-118.179767,Long Beach City,5,Active,1,4,LB,City of Long Beach Health and Human Services,City,2525 Grand Ave,Long Beach,Hans Tritten,90815,NULL,NULL,NULL,Jackie Hampton,Los Angeles,1,4,Los Angeles
1,RB4,436,B-

Columns: ['DURATION CALCULATED', 'Regional Board', 'Advisory id', 'HistoricalStationName', 'prefix', 'mile', 'DateofAdvisory', 'TimeofAdvisory', 'DateOpened', 'TimeOpened', 'AdvisoryType', 'AdvisoryCause', 'Description', 'Extent', 'StartofExtent', 'StartOfExtentType', 'EndofExtent', 'EndOfExtentType', 'Source', 'Advisory Comments', 'DescriptionofArea', 'ActionBy', 'AffectedLengthUnit', 'SubstanceName', 'SubstanceVolume', 'SubstanceUnit', 'AdditionalAction', 'ActionDate', 'ActionDescription', 'County', 'User_id', 'Enterococcus', 'Fecal Coliforms', 'Total Coliforms', 'E.Coli', 'Ratio', 'Advisory Record Create Date', 'Station_id', 'Station_Agency_id', 'BeachName_id', 'Station_Name', 'Station_Description', 'AgencyStationIdentifier', 'Station_UpperLat', 'Station_UpperLon', 'Station_LowerLat', 'Station_LowerLon', 'SwimSeason', 'OffSeason', 'AsNeededSeason', 'CountyCode', 'CountyName', 'Status', 'UpdateDate', 'Advisories Insert Date', 'pointZero', 'Datum', 'BeachProfileId', 'Beach_AgencyName'

In [0]:
ca_beach_results = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("samplingRatio", "0.0001")
    .load("s3://toxictide-raw/sources/ca_beach_water_quality/raw/*/monitoring_results/*.csv")
)

display(ca_beach_results.limit(20))
print("Columns:", ca_beach_results.columns)

Regional Board,RESULTS id,SampleDate,StartTime,StartTimeInd,Parameter,QualifierId,Qualifier,QualifierDescription,Result,Unit,LabCode_id,AnalysisMethodId,AnalysisMethod,SampleTypeId,SampleType,Results Comments,County,User_id,upload_ts,Result Record Create Date,LabID,StormDrainFlow,Weather,TidalHeight,SurfHeight,Turbidity,WaterColor,Odor,Distance,Direction,RL,MDL,Dilution,Station_id,Station_Agency_id,BeachName_id,Station_Name,Station_Description,AgencyStationIdentifier,Station_UpperLat,Station_UpperLon,Station_LowerLat,Station_LowerLon,SwimSeason,OffSeason,AsNeededSeason,CountyCode,CountyName,Status,UpdateDate,Results Insert Date,pointZero,Beach_AgencyName,Beach_Name,Description,BeachType,Beach Length,AttendanceWinter,AttendanceSummer,NearestCityName,WaterBodyName,WaterBodyClass,AB411Beach,USEPAID,BeachAccess,SwimSeasonLength,WaterBodyType,WaterShedName,Beach_UpperLat,Beach_UpperLon,Beach_LowerLat,Beach_LowerLon,Beach_County,Beach_User_id,Beach_Status,CountAsBeach,Agency_id,Agency_Code,Agency_Name,Agency_Jurisdiction,StreetAddress,City,Results Contact Person,Zip,EstMonCost,EstCoastlineAdvisoryCost,EstTotalCost,Agency_Comments,Agency_County,Agency_User_id,Regional Board Number,Regional Board Name
RB2,249714,2005-08-02,2026-04-19T00:00:00.000Z,0,E. Coli,1,=,Equal to,20,CFU/100ml,8,14,Colilert-18,1,Results,null,Marin,12,null,2005-08-02,null,null,null,null,null,null,null,null,null,null,0,0,1,255,8,240,CHICKEN RANCH,Chicken Ranch Beach at Creek,CHICKEN RANCH,38.109741,-122.8646,38.10907,-122.8646,None(not sampled),None(not sampled),None(not sampled),MN,Marin,Active,2018-03-28,2015-08-19,1,County of Marin Environmental Health Services,Chicken Ranch,Chicken Ranch Beach at Creek,ROCKY,0.53,0,0,Iverness,Pacific Ocean,Estuarine,Yes,CA938432,PUBLIC,7,"Sound, Bay, or Inlet",Tomales Bay-Drakes Bay,38.108533,-122.86405,38.114608,-122.869852,Marin,12,Active,1,8,MN,County of Marin Environmental Health Services,County,3501 Civic Center Drive Room 236,San Rafel,Robert Turner,94903,NotAvailable,NotAvailable,$50000-$99999,DUNS=078787744,Marin,1,2,San Francisco Bay
RB2,250255,2005-10-25,2026-04-19T00:00:00.000Z,0,E. Coli,1,=,Equal to,31,CFU/100ml,8,14,Colilert-18,1,Results,null,Marin,12,null,2005-10-25,null,null,null,null,null,null,null,null,null,null,0,0,1,255,8,240,CHICKEN RANCH,Chicken Ranch Beach at Creek,CHICKEN RANCH,38.109741,-122.8646,38.10907,-122.8646,None(not sampled),None(not sampled),None(not sampled),MN,Marin,Active,2018-03-28,2015-08-19,1,County of Marin Environmental Health Services,Chicken Ranch,Chicken Ranch Beach at Creek,ROCKY,0.53,0,0,Iverness,Pacific Ocean,Estuarine,Yes,CA938432,PUBLIC,7,"Sound, Bay, or Inlet",Tomales Bay-Drakes Bay,38.108533,-122.86405,38.114608,-122.869852,Marin,12,Active,1,8,MN,County of Marin Environmental Health Services,County,3501 Civic Center Drive Room 236,San Rafel,Robert Turner,94903,NotAvailable,NotAvailable,$50000-$99999,DUNS=078787744,Marin,1,2,San Francisco Bay
RB2,250256,2005-10-18,2026-04-19T00:00:00.000Z,0,E. Coli,1,=,Equal to,10,CFU/100ml,8,14,Colilert-18,1,Results,null,Marin,12,null,2005-10-18,null,null,null,null,null,null,null,null,null,null,0,0,1,255,8,240,CHICKEN RANCH,Chicken Ranch Beach at Creek,CHICKEN RANCH,38.109741,-122.8646,38.10907,-122.8646,None(not sampled),None(not sampled),None(not sampled),MN,Marin,Active,2018-03-28,2015-08-19,1,County of Marin Environmental Health Services,Chicken Ranch,Chicken Ranch Beach at Creek,ROCKY,0.53,0,0,Iverness,Pacific Ocean,Estuarine,Yes,CA938432,PUBLIC,7,"Sound, Bay, or Inlet",Tomales Bay-Drakes Bay,38.108533,-122.86405,38.114608,-122.869852,Marin,12,Active,1,8,MN,County of Marin Environmental Health Services,County,3501 Civic Center Drive Room 236,San Rafel,Robert Turner,94903,NotAvailable,NotAvailable,$50000-$99999,DUNS=078787744,Marin,1,2,San Francisco Bay
RB2,250257,2005-10-11,2026-04-19T00:00:00.000Z,0,E. Coli,1,=,Equal to,9,CFU/100ml,8,14,Colilert-18,1,Results,null,Marin,12,null,2005-10-11,null,null,null,null,null,null,null,null,null,null,

Columns: ['Regional Board', 'RESULTS id', 'SampleDate', 'StartTime', 'StartTimeInd', 'Parameter', 'QualifierId', 'Qualifier', 'QualifierDescription', 'Result', 'Unit', 'LabCode_id', 'AnalysisMethodId', 'AnalysisMethod', 'SampleTypeId', 'SampleType', 'Results Comments', 'County', 'User_id', 'upload_ts', 'Result Record Create Date', 'LabID', 'StormDrainFlow', 'Weather', 'TidalHeight', 'SurfHeight', 'Turbidity', 'WaterColor', 'Odor', 'Distance', 'Direction', 'RL', 'MDL', 'Dilution', 'Station_id', 'Station_Agency_id', 'BeachName_id', 'Station_Name', 'Station_Description', 'AgencyStationIdentifier', 'Station_UpperLat', 'Station_UpperLon', 'Station_LowerLat', 'Station_LowerLon', 'SwimSeason', 'OffSeason', 'AsNeededSeason', 'CountyCode', 'CountyName', 'Status', 'UpdateDate', 'Results Insert Date', 'pointZero', 'Beach_AgencyName', 'Beach_Name', 'Description', 'BeachType', 'Beach Length', 'AttendanceWinter', 'AttendanceSummer', 'NearestCityName', 'WaterBodyName', 'WaterBodyClass', 'AB411Beach',

In [0]:
ca_beach_stations_clean, ca_beach_stations_colmap = sanitize_columns(ca_beach_stations)
ca_beach_postings_clean, ca_beach_postings_colmap = sanitize_columns(ca_beach_postings)
ca_beach_results_clean, ca_beach_results_colmap = sanitize_columns(ca_beach_results)

display(spark.createDataFrame([(k, v) for k, v in ca_beach_stations_colmap.items()], ["original", "sanitized"]))
display(spark.createDataFrame([(k, v) for k, v in ca_beach_postings_colmap.items()], ["original", "sanitized"]))
display(spark.createDataFrame([(k, v) for k, v in ca_beach_results_colmap.items()], ["original", "sanitized"]))

ca_beach_stations_clean.write.mode("overwrite").saveAsTable("toxictide.bronze.ca_beach_monitoring_stations_raw")
ca_beach_postings_clean.write.mode("overwrite").saveAsTable("toxictide.bronze.ca_beach_postings_closures_raw")
ca_beach_results_clean.write.mode("overwrite").saveAsTable("toxictide.bronze.ca_beach_monitoring_results_raw")

original,sanitized
Regional Board,regional_board
Station_id,station_id
Station_Agency_id,station_agency_id
BeachName_id,beachname_id
Station_Name,station_name
Station_Description,station_description
AgencyStationIdentifier,agencystationidentifier
Station_UpperLat,station_upperlat
Station_UpperLon,station_upperlon
Station_LowerLat,station_lowerlat


original,sanitized
DURATION CALCULATED,duration_calculated
Regional Board,regional_board
Advisory id,advisory_id
HistoricalStationName,historicalstationname
prefix,prefix
mile,mile
DateofAdvisory,dateofadvisory
TimeofAdvisory,timeofadvisory
DateOpened,dateopened
TimeOpened,timeopened


original,sanitized
Regional Board,regional_board
RESULTS id,results_id
SampleDate,sampledate
StartTime,starttime
StartTimeInd,starttimeind
Parameter,parameter
QualifierId,qualifierid
Qualifier,qualifier
QualifierDescription,qualifierdescription
Result,result


In [0]:
cce_summaries = (
    summary_inventory
    .filter(F.col("source_name") == "cce_moorings")
)

display(
    cce_summaries.select(
        "source_group",
        "filename",
        "file_type",
        "size_bytes",
        # "dimensions",
        # "coordinates",
        # "data_variables"
    )
)

source_group,filename,file_type,size_bytes


In [0]:
cce_summaries_enriched = (
    cce_summaries
    .withColumn(
        "sensor_family",
        F.regexp_extract(F.col("filename"), r"_(?:D|P|M)_(.+)\.nc$", 1)
    )
)

display(
    cce_summaries_enriched
    .groupBy("source_group", "sensor_family")
    .count()
    .orderBy("source_group", "sensor_family")
)

source_group,sensor_family,count


In [0]:
rep_families = ["CHL", "Meteorology", "ADCP", "CTD", "AQUADOPP"]

w = Window.partitionBy("source_group", "sensor_family").orderBy("size_bytes")

cce_representatives = (
    cce_summaries_enriched
    .filter(F.col("sensor_family").isin(rep_families))
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("source_group", "sensor_family", "filename", "size_bytes")#, "data_variables", "dimensions")
    .orderBy("source_group", "sensor_family")
)

display(cce_representatives)

source_group,sensor_family,filename,size_bytes


In [0]:
cce_summaries_enriched.write.mode("overwrite").saveAsTable("toxictide.bronze.cce_moorings_summary_inventory")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6904815970628407>, line 1
----> 1 cce_summaries_enriched.write.mode("overwrite").saveAsTable("toxictide.bronze.cce_moorings_summary_inventory")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   1537 self.

In [0]:
argo_manifests = load_manifest_inventory("easyoneargo")

display(
    argo_manifests.select(
        "source_name",
        "run_id",
        "run_date",
        "status",
        F.size("records").alias("n_records"),
        "notes"
    )
)

source_name,run_id,run_date,status,n_records,notes
easyoneargo,2026-04-19T07-48-40Z,2026-04-19,completed,5,List()
easyoneargo,2026-04-19T07-46-04Z,2026-04-19,completed,5,List()
easyoneargo,2026-04-19T03-36-42Z,2026-04-19,completed,5,List()
easyoneargo,2026-04-19T07-31-08Z,2026-04-19,completed,5,List()


In [0]:
argo_inventory = raw_inventory.filter(F.col("source_name") == "easyoneargo")
display(argo_inventory.orderBy(F.desc("length")))

source_name,path,length,modificationTime
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/127233.tar.gz,7969262537,2026-04-19T08:00:49.000Z
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/126471.tar.gz,7907834785,2026-04-19T08:00:34.000Z
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/127234.tar.gz,1595051070,2026-04-19T07:50:26.000Z
easyoneargo,s3://toxictide-raw/sources/easyoneargo/raw/run_date=2026-04-19/126470.tar.gz,1586383338,2026-04-19T07:50:47.000Z
